<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>

<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# Network Protocols and Quality

The previous notebook explained how packets find a destination. But one Duckiedrone can provide several services through the same IP address: an SSH shell, a dashboard, and access to sensor data.

These services also place different demands on the connection. An SSH terminal may remain usable while a camera stream freezes or arrives late.

This notebook explains how protocols and ports distinguish services, and how delay, capacity, and packet loss affect their use.

## From a name to a service

There are several steps between a hostname and a network service. Figure 1 illustrates the process.

<figure id="figure-1" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
    hostname
      |
      v
    address lookup
      |
      v
    route
      |
      v
    reachable Duckiedrone
      |
      v
    protocol and port
      |
      v
    service
  </pre>
  <figcaption>Figure 1: Steps from a hostname to a network service.</figcaption>
</figure>


When you run:

```bash
ssh duckie@amelia.local
```

the SSH client normally connects to TCP port `22` at an address resolved from `amelia.local`. The username `duckie` selects the account; it does not select the network port.

An **endpoint** combines an address, transport protocol, and port. For example:

```text
Address:   192.168.1.201
Transport: TCP
Port:      22
```

This describes the usual SSH endpoint at Amelia’s previously recorded address. The address alone does not distinguish SSH from another service.

The SSH program on your base station is the **client**. The SSH server on the robot accepts connections and handles requests. Your browser and the robot’s dashboard follow the same **client-server model**.

When several processes cooperate across computers—for example, a robot producing measurements and a base station displaying them—they form a **distributed application**.

### Agree on what the data means

Delivering bytes is not enough: the receiving program must understand them.

Imagine a robot reports an altitude value of `0.8`. Is that metres or centimetres? When was it measured?

The application’s **data contract** specifies fields and meanings, including units and timestamps. Its **application protocol** defines the messages exchanged, while an **Application Programming Interface (API)** defines how software accesses the functionality. TCP or UDP transports the data without defining its meaning.

## Ports and transport protocols

A **port** helps the operating system deliver network traffic to the appropriate application. Port numbers belong to a transport protocol: TCP port `22` and UDP port `22` are different endpoints.

| Protocol | What it provides | Example |
| --- | --- | --- |
| **TCP — Transmission Control Protocol** | An ordered, reliable byte stream, with retransmission of lost data | SSH and HTTP/1.1 dashboard connections |
| **UDP — User Datagram Protocol** | Individual messages, without built-in guarantees of delivery or ordering | Multicast DNS queries for `.local` names |

TCP’s reliability is useful for an SSH session: missing or reordered bytes must not silently change the command you typed. However, retransmission can introduce delay, and a connection can still fail.

UDP leaves recovery decisions to the application. Software using UDP may implement retries or other reliability mechanisms itself. The [TCP](https://www.rfc-editor.org/rfc/rfc9293.html) and [UDP](https://www.rfc-editor.org/rfc/rfc768.html) specifications describe these differences.

`ping` uses **Internet Control Message Protocol (ICMP)**, not TCP or UDP. Its echo requests have no service port. A successful ping therefore does not establish that SSH or the dashboard is available.

### Try it: distinguish a reply from a listener

In your Linux Workspace or Duckiedrone shell, run:

```bash
ping -c 3 127.0.0.1
ss -ltn
```

The first command sends three loopback echo requests. The second lists listening TCP sockets: `-l` selects listeners, `-t` selects TCP, and `-n` keeps addresses and ports numeric.

An illustrative row is:

```text
State  Recv-Q Send-Q Local Address:Port Peer Address:Port
LISTEN 0      128    0.0.0.0:22         0.0.0.0:*
```

This shows a TCP listener on port `22`, bound to all local IPv4 addresses in that environment. It does not verify that the listener is SSH or that another computer can reach it. See the [`ss` manual](https://man7.org/linux/man-pages/man8/ss.8.html).

Your listing may contain different ports or no listeners. Record one endpoint if available, then answer:

**Would successful loopback ping replies guarantee that this TCP listener exists?**

<details>
<summary>Reveal answer</summary>

No. Loopback ping tests ICMP within the current environment. It can succeed even when no TCP services are listening.

</details>

The service-binding and diagnostics notebooks develop these checks further.

## Network quality

A connection can work without working well enough for the task. Four measurements help describe the difference:

| Measurement | Meaning | Effect you might notice |
| --- | --- | --- |
| **Latency** | Time taken for data to travel | A delay between typing in SSH and seeing the response |
| **Bandwidth** | The connection’s data-carrying capacity | Limited capacity can constrain camera data rates |
| **Jitter** | Variation in packet delay | Updates arrive at uneven intervals |
| **Packet loss** | Some packets fail to arrive | Retransmission delays or missing application data |

`ping` reports **round-trip time**: the request’s journey out and the reply’s journey back. It does not directly measure one-way delay or available bandwidth.

Consider these illustrative round-trip times:

```text
Connection A: 10 ms, 11 ms, 10 ms, 11 ms
Connection B:  2 ms,  2 ms,  2 ms, 36 ms
```

Both average `10.5 ms`, but B varies much more. That variation can matter for timely robot feedback even when the average looks acceptable.

Likewise, an interactive SSH session carries little data compared with a camera stream. A usable terminal does not prove there is enough available capacity for smooth video. Actual achieved data rate, often called **throughput**, can be lower than the connection’s nominal bandwidth.

Wi-Fi conditions vary with distance, obstacles, interference, and other traffic. Robot processing or camera software can also cause delays. Record the symptom and circumstances before attributing it to the network; no artificial load test is needed for this exercise.

## How a packet crosses a network

The identifiers introduced so far work at different layers. For SSH:

<figure id="figure-2" class="lx-figure">
  <pre style="display:inline-block; margin:0; text-align:left;">
SSH: commands and terminal output
             |
TCP: source and destination ports
             |
IP: source and destination addresses
             |
Ethernet or Wi-Fi: delivery over the local link
  </pre>
  <figcaption>Figure 2: Layers carrying an SSH connection.</figcaption>
</figure>

Each layer adds information needed for its part of delivery. This wrapping is called **encapsulation**.

For an Ethernet example, a packet addressed to a local robot is carried in a frame addressed to that robot’s MAC address. For an off-subnet destination, the frame instead targets the next-hop router. Wi-Fi also uses local-link addressing, with the access point participating in delivery.

IPv4 uses **Address Resolution Protocol (ARP)** to learn the MAC address associated with a local next-hop IP address. Routers then forward packets over subsequent links, using new link-layer frames.

This explains why an IP address and route are necessary but not sufficient: local delivery, the transport connection, and the application must also work.

## Further reading

The IETF specification for [packet delay variation](https://www.rfc-editor.org/rfc/rfc3393.html) provides technical detail on jitter.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
